<a href="https://colab.research.google.com/github/khaled-cheour/LandiGlobal_Invoice/blob/main/%F0%9F%A7%A0%20Code%20ML%20%E2%80%94%20Pr%C3%A9diction%20de%20retard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install requests pandas

In [4]:
import requests
import pandas as pd

# ── Configuration ──────────────────────────────────────
URL      = "https://api.asinaria.xyz/powerbi/json/piVT9Vo"
EMAIL    = "khaled.cheour@landiglobal.com"
TOKEN    = "VOTRE_TOKEN_ICI"   # ← remplace ici, ne partage jamais ce token

# ── Connexion à l'API ──────────────────────────────────
headers = {
    "Accept": "application/json"
}
auth = (EMAIL, TOKEN)

# ── Requête pour récupérer les tickets ────────────────
response = requests.get(URL, headers=headers, auth=auth)

if response.status_code == 200:
    data = response.json()
    print("✅ Connexion réussie !")
    print(f"   Nombre de tickets : {len(data)}")
else:
    print(f"❌ Erreur {response.status_code} : {response.text}")
    data = []

# ── Transformation en DataFrame pandas ────────────────
if data:
    df = pd.DataFrame(data)
    print("\n📋 Colonnes disponibles :")
    print(df.columns.tolist())
    print("\n🔍 Aperçu des données :")
    print(df.head())

    # ── Sauvegarde CSV pour la suite ──────────────────
    df.to_csv("jira_tickets.csv", index=False)
    print("\n💾 Fichier sauvegardé : jira_tickets.csv")


✅ Connexion réussie !
   Nombre de tickets : 1126

📋 Colonnes disponibles :
['Id', 'Key', 'Project Key', 'Project Name', 'Parent', 'Status Category Changed', 'Issue Type (Name)', 'Issue Type (Description)', 'Work Ratio', 'Created', 'Priority', 'Labels', 'Rank', 'Updated', 'Status', 'Start date', 'Summary', 'Creator', 'Reporter', 'Due date', 'Votes', 'Assignee', 'Origin Region', 'Opportunity Name | Labels', 'Opportunity Name | Keys', 'Opportunity Name | Object IDs', 'Project type', 'Project confidence', 'Device management', 'Product Family', 'CustomerCompanyName', 'OpportunityName', 'Customer Company Name | Labels', 'Customer Company Name | Keys', 'Customer Company Name | Object IDs', 'Customer Type', 'Target end', 'Sales Manager', 'Resolution', 'Resolved', 'Third party MDM name', 'Customer Company Name.', 'Key injection', 'Processor', 'Acquirer', 'Device - additional information', 'Gateway', 'L3 certification name', 'L3 certification date', 'Next Step', 'Result', 'Previous status', 'La

In [5]:
pip install pandas scikit-learn matplotlib seaborn joblib

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("jira_tickets.csv")

# Convertir les dates
df['Created']  = pd.to_datetime(df['Created'],  utc=True, errors='coerce')
df['Due date'] = pd.to_datetime(df['Due date'], utc=True, errors='coerce')
df['Resolved'] = pd.to_datetime(df['Resolved'], utc=True, errors='coerce')

# Garder seulement les tickets avec due date et resolved
df = df.dropna(subset=['Due date', 'Resolved'])

# Créer la cible : 1 = retard, 0 = à temps
df['en_retard'] = (df['Resolved'] > df['Due date']).astype(int)

# Durée prévue (jours)
df['duree_prevue'] = (df['Due date'] - df['Created']).dt.days

print(f"✅ Tickets avec due date + resolved : {len(df)}")
print(f"   Tickets en retard : {df['en_retard'].sum()} ({df['en_retard'].mean()*100:.1f}%)")
print(f"   Tickets à temps   : {(df['en_retard']==0).sum()}")

✅ Tickets avec due date + resolved : 552
   Tickets en retard : 384 (69.6%)
   Tickets à temps   : 168


In [8]:
import pandas as pd

df = pd.read_csv("jira_tickets.csv")

# Convertir les dates
df['Created']  = pd.to_datetime(df['Created'],  utc=True, errors='coerce')
df['Due date'] = pd.to_datetime(df['Due date'], utc=True, errors='coerce')
df['Resolved'] = pd.to_datetime(df['Resolved'], utc=True, errors='coerce')
df['Target end'] = pd.to_datetime(df['Target end'], utc=True, errors='coerce')

# Voir combien de valeurs remplies dans chaque colonne utile
cols = ['Due date', 'Resolved', 'Target end', 'Priority',
        'Issue Type (Name)', 'Customer Region',
        'Project confidence', 'Status', 'duree_prevue']

print("📊 Remplissage des colonnes (sur 1126 tickets) :")
print("-" * 45)
for col in cols:
    if col == 'duree_prevue':
        df['duree_prevue'] = (df['Due date'] - df['Created']).dt.days
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    barre = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"{col:<25} {barre}  {non_null:>4} ({pct:.0f}%)")

📊 Remplissage des colonnes (sur 1126 tickets) :
---------------------------------------------
Due date                  ████████████████░░░░   932 (83%)
Resolved                  ██████████░░░░░░░░░░   598 (53%)
Target end                ████████░░░░░░░░░░░░   452 (40%)
Priority                  ████████████████████  1126 (100%)
Issue Type (Name)         ████████████████████  1126 (100%)
Customer Region           ░░░░░░░░░░░░░░░░░░░░     1 (0%)
Project confidence        ██████░░░░░░░░░░░░░░   373 (33%)
Status                    ████████████████████  1126 (100%)
duree_prevue              ████████████████░░░░   932 (83%)


In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("jira_tickets.csv")

# Convertir les dates
df['Created']    = pd.to_datetime(df['Created'],    utc=True, errors='coerce')
df['Due date']   = pd.to_datetime(df['Due date'],   utc=True, errors='coerce')
df['Target end'] = pd.to_datetime(df['Target end'], utc=True, errors='coerce')
df['Updated']    = pd.to_datetime(df['Updated'],    utc=True, errors='coerce')

# Cible : retard = Updated après Due date (pour les tickets fermés)
df = df.dropna(subset=['Due date'])
df['en_retard'] = (df['Updated'] > df['Due date']).astype(int)

# Feature : durée prévue en jours
df['duree_prevue'] = (df['Due date'] - df['Created']).dt.days

# Feature : jour de la semaine de création (0=lundi ... 6=dimanche)
df['jour_creation'] = df['Created'].dt.dayofweek

# Feature : mois de création
df['mois_creation'] = df['Created'].dt.month

print(f"✅ Tickets disponibles : {len(df)}")
print(f"   En retard : {df['en_retard'].sum()} ({df['en_retard'].mean()*100:.1f}%)")
print(f"   À temps   : {(df['en_retard']==0).sum()} ({(1-df['en_retard'].mean())*100:.1f}%)")

✅ Tickets disponibles : 932
   En retard : 584 (62.7%)
   À temps   : 348 (37.3%)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# Features adaptées aux colonnes disponibles à 100%
features = ['Priority', 'Issue Type (Name)', 'Status',
            'duree_prevue', 'jour_creation', 'mois_creation']

df_ml = df[features + ['en_retard']].dropna()
print(f"✅ Lignes pour le modèle : {len(df_ml)}")

# Encoder les colonnes texte
le = {}
for col in ['Priority', 'Issue Type (Name)', 'Status']:
    le[col] = LabelEncoder()
    df_ml = df_ml.copy()
    df_ml[col] = le[col].fit_transform(df_ml[col].astype(str))

X = df_ml[features]
y = df_ml['en_retard']

# Split train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entraînement
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Évaluation
y_pred = model.predict(X_test)
print("\n📊 Performance du modèle :")
print(classification_report(y_test, y_pred))

# Importance des features
print("🔍 Importance des features :")
for feat, imp in sorted(zip(features, model.feature_importances_),
                         key=lambda x: -x[1]):
    barre = "█" * int(imp * 40)
    print(f"  {feat:<25} {barre}  {imp*100:.1f}%")

# Sauvegarde
joblib.dump(model, 'model_retard.pkl')
joblib.dump(le, 'encoders.pkl')
print("\n💾 Modèle sauvegardé !")

✅ Lignes pour le modèle : 932

📊 Performance du modèle :
              precision    recall  f1-score   support

           0       0.82      0.92      0.87        60
           1       0.96      0.91      0.93       127

    accuracy                           0.91       187
   macro avg       0.89      0.91      0.90       187
weighted avg       0.91      0.91      0.91       187

🔍 Importance des features :
  Status                    █████████████████  43.6%
  duree_prevue              ██████████  25.4%
  Issue Type (Name)         ████  12.0%
  mois_creation             ████  10.3%
  jour_creation             ██  5.7%
  Priority                  █  2.9%

💾 Modèle sauvegardé !


In [11]:
import joblib, pandas as pd

model = joblib.load('model_retard.pkl')
le    = joblib.load('encoders.pkl')

nouveau = {
    'Priority':           'High',
    'Issue Type (Name)':  'Sub-task',
    'Status':             'In Progress',
    'duree_prevue':       45,
    'jour_creation':      0,
    'mois_creation':      4
}

df_test = pd.DataFrame([nouveau])
for col in ['Priority', 'Issue Type (Name)', 'Status']:
    df_test[col] = le[col].transform(df_test[col].astype(str))

proba = model.predict_proba(df_test)[0][1]
print(f"🎯 Risque de retard : {proba*100:.1f}%")
if proba > 0.5:
    print("⚠️  Ce ticket risque d'être en retard !")
else:
    print("✅  Ce ticket devrait être livré à temps.")

🎯 Risque de retard : 9.0%
✅  Ce ticket devrait être livré à temps.
